In [ ]:
'''
Runs all PAI experiments for AlphaFold protein residue disorder rate (finite-population).

Remarks:
1. PAI is by definition sequential.
2. Because the ML model is fixed, SAI will be run in batch mode (i.e., no budget-stabilization).
3. We'll use Zrnic & Candes's f_t(x) = p_t(x) and u_t(x) = 2\min\{ p_t(x), 1 - p_t(x) \} with tau=0.5.
'''
import torch
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
import os, csv

# set a device + specify number of seeds, uniform-mixing & CI target coverage + spread of budgets
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, TAU, ALPHA = 1000, 0.5, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)

# filename for this simulation
fname = f"alphafold_pai.csv"

# dataframe to store all of our results
columns = ["dataset", "budget", "tau", "method", "seed", "thetahat", "coverage", "width", "n_labels"]
if fname not in os.listdir("../logs"):
    with open(f"../logs/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")

In [ ]:
#### LOAD IN OUR DATA

# for alphafold, just need y_test and fhat_test (fixed AlphaFold-derived ML predictions)
fxs = torch.load("../datasets/alphafold/fhat_test.pt").to(device) # (10802,)
ys = torch.load("../datasets/alphafold/y_test.pt").to(device) # (10802,)

# get our finite-population size
N = ys.shape[0]

# can precompute our uncertainty scores: u_t \propto ( p_t(x), 1 - p_t(x) )
u_ts = torch.minimum(torch.tensor(1.0, device=device) - fxs, fxs)

In [ ]:
#### PRO-ACTIVE INFERENCE SAMPLING

# iterate through our budgets
for budget_idx, budget in enumerate(BUDGET_PROPS):
    
    # how many labels can we collect?
    n = int(budget * N)
    
    #### initialize data structures
    
    # set a seed for reproducibility
    torch.random.manual_seed(858 + budget_idx)

    # data structure of which samples have already been observed?
    already_sampled = torch.zeros(size=(N_SEEDS, N), dtype=bool, device=device)

    # recurring sum (WITHOUT NORMALIZATION) of thetahat (need div by n) and sigmahat^2 (need div by nN^2)
    thetahats_unnormalized = torch.zeros(size=(N_SEEDS, 1), device=device, dtype=torch.float32)
    sigmahats_sq_unnormalized = torch.zeros(size=(N_SEEDS, 1), device=device, dtype=torch.float32)
    
    #### adaptive sampling rounds
    
    # loop thru our budget
    for t in tqdm(range(n), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}"):
        
        #### PART 1. computing q_t(j) probabilities + sampling probabilities

        # a. zero out previously observed entries, normalize, mix with tau-governed uniform
        h_js = u_ts * ~already_sampled # (N_SEEDS, N)
        h_js /= h_js.sum(axis=1, keepdim=True) # (N_SEEDS, N)
        q_js = (TAU * ~already_sampled / (N - t)) + ((1.0 - TAU) * h_js) #(N_SEEDS, N)

        # b. which datapoint are we sampling?
        I_t = torch.multinomial(input=q_js, num_samples=1)
        
        #### PART 2: Updating Our PAI Estimators

        # c. get the queried values + build AIPW correction
        y_Is = ys[I_t] # (N_SEEDS, 1)
        f_Is = fxs[I_t] # (N_SEEDS, 1)
        q_Is = torch.gather(q_js, dim=1, index=I_t) # (N_SEEDS, 1)
        aipw_correction_Is = (y_Is - f_Is) / q_Is
        
        # d. building the Ahat_n (without the / nN^2)
        sigmahats_sq_unnormalized += (aipw_correction_Is ** 2) # first term
        
        # e. building the Bhat_n (also without the / nN^2)
        if t > 0:
            t2 = (N * thetahats_unnormalized / t)\
            - (ys * already_sampled).sum(dim=1, keepdim=True)\
            - (fxs * ~already_sampled).sum(dim=1, keepdim=True)
            sigmahats_sq_unnormalized -= (t2 ** 2)
            
        # f. add in the bias-correction term Chat_n for all but the last term (also without the / nN^2)
        if t < n - 1:
            t3 = (aipw_correction_Is ** 2)
            t3 *= ((1.0 / torch.arange(t+1, n, device=device, dtype=torch.float32)) ** 2).sum()
            sigmahats_sq_unnormalized += t3
            
        # g. compute our phi_t and increment our estimator (UNNORMALIZED)
        phi_ts = (
            (fxs * ~already_sampled).sum(dim=1, keepdim=True) +\
            (ys * already_sampled).sum(dim=1, keepdim=True) +\
            aipw_correction_Is
        ) / N
        thetahats_unnormalized += phi_ts
        
        # h. update our observed indices
        already_sampled.scatter_(dim=1, index=I_t, src=torch.full_like(I_t, fill_value=True, dtype=bool))
        
    #### PART 3: combining + normalizing to make PAI estimates
    
    # dummy tensor for brevity
    ZERO = torch.tensor(0.0, device=device)

    # get our mean estimate + variance estimate
    thetahats = thetahats_unnormalized / n
    sigmahats_sq = torch.maximum(sigmahats_sq_unnormalized, ZERO) / (n * (N ** 2))
    
    # get our z-score + the lower/upper limits of our interval
    z_score = norm.ppf(1 - (ALPHA / 2))
    ubs = thetahats + z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
    lbs = thetahats - z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
    widths = (ubs - lbs).cpu().numpy()
    coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()
    
    # for file writing purposes, convert thetahats
    thetahats = thetahats.cpu().numpy()
    
    # format results + directly write to file
    results = [
        ["alphafold", budget, TAU, "pai", seed, 
         thetahats[seed][0], coverages[seed][0], widths[seed][0], n
        ] for seed in range(N_SEEDS)]
    with open(f"../logs/{fname}", "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerows(results)